# 06. Ingeniería de Features Híbridas

**Objetivo:** construir la matriz final de features (`feat_*`, `feat_niega_*`, `rule_medication_*`, `sent_*`, `ctx_<backbone>_*`).
**Entradas (inputs):** `data/dataset_denoised.csv`, `data/processed/gemini_extraction.json` (opcional), reglas congeladas `Concept_Core` + `Concept_PY`, selección de backbone desde 04c (opcional).
**Salidas (outputs):** `data/processed/fe_<run_id>_core/features_core.parquet`, `data/processed/fe_<run_id>_py/features_py.parquet`, `*_feature_summary.csv`, `*_config.json`.
**Notebook anterior:** `notebooks/pipeline/03_denoising_reglas_core.ipynb` y `notebooks/pipeline/04c_linea_base_transformers.ipynb`.
**Notebook siguiente:** `notebooks/pipeline/07_entrenamiento_modelos_hibridos.ipynb`.

> **Consumo explícito:** el backbone contextual usa `BETO` por defecto para mantener alineación con la comparación controlada del híbrido; puede forzarse por entorno con `FE_TEXT_BACKBONE`, y `auto` queda disponible solo como override explícito para consumir la selección de `04c`. `04b` no alimenta este bloque: `TF-IDF` funciona como baseline textual aislado.


## Técnicas, herramientas y librerías de esta etapa

- **Técnica principal:** construcción de features híbridas a partir de reglas clínicas, `late fusion` sintomático, sentimiento y embeddings contextuales `ctx_<backbone>_*`.
- **Herramientas/librerías:** fork clínico `Spanish_Psych_Phenotyping_PY`, `spaCy`, `medSpaCy`, `pysentimiento`, `transformers`, `torch`, `pandas`, `numpy`, `tqdm`.
- **Por qué es adecuada aquí:** el proyecto necesita combinar trazabilidad clínica con una señal contextual reusable. Esta etapa permite mantener separadas las familias de features y dejar claro qué bloque aporta qué tipo de evidencia.
- **Limitación:** sentimiento y LLM son componentes auxiliares; no deben sobreinterpretarse como núcleo clínico. Los embeddings contextuales aportan contexto, pero siguen siendo menos auditables que las reglas.
- **Alternativa si se quisiera otra arquitectura:** una fusión end-to-end más compleja podría modelar interacciones más ricas, pero perdería auditabilidad y complicaría el cierre metodológico ya congelado.


## Banderas de ejecución y trazabilidad (06)

Este notebook define explícitamente el estado de la corrida para que 07 pueda reproducir y auditar cada variante.

**Flags principales (entorno):**
- `FE_RUN_ID`: identificador de corrida de features.
- `FE_USE_LLM = auto | 1 | 0`: controla la normalización semántica con LLM.
  - `auto`: usa LLM solo si existe `gemini_extraction.json`.
  - `1`: fuerza LLM y falla si falta el archivo.
  - `0`: desactiva LLM aunque exista el archivo.
- `FE_COMPUTE_SENTIMENT = 1 | 0`: activa/desactiva `sent_*`.
- `FE_COMPUTE_CONTEXT = 1 | 0`: activa/desactiva embeddings contextuales `ctx_<backbone>_*`.
- `FE_TEXT_BACKBONE = beto | auto | roberta_clinical | roberta_biomedical`: backbone textual contextual.
  - por defecto usa `beto`, porque la comparación controlada del híbrido retuvo ese backbone en `dev`;
  - `auto` consume la selección vigente de `04c` solo si querés esa conducta de forma explícita;
  - `04b` no interviene en esta decisión.
- `FE_CACHE_KEY`: permite reutilizar cachés entre corridas.

**Familias de features exportadas:**
- `feat_*` y `feat_niega_*`: señal clínica final (reglas + late fusion con LLM cuando aplica).
- `rule_*` y `niega_*`: evidencia simbólica explícita para auditoría/ablación.
- `rule_medication_*`: evidencia terapéutica separada, sin fusión diagnóstica directa con LLM.
- `sent_*`: señal afectiva complementaria.
- `ctx_<backbone>_*`: embeddings contextuales del backbone seleccionado.


In [ ]:
# =====================
# Importaciones y entorno
# =====================
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

current_dir = Path.cwd()
REPO_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
sys.path.insert(0, str(REPO_ROOT))

try:
    from notebooks.utils_shared import setup_paths, keep_entity, guess_text_col
except Exception:
    from utils_shared import setup_paths, keep_entity, guess_text_col

paths = setup_paths()
DATA_PATH = paths["DATA_PATH"]
SPLITS_PATH = paths["SPLITS_PATH"]
FORK_PATH = paths["FORK_PATH"]
PROCESSED_PATH = DATA_PATH / "processed"
OUTPUTS_PATH = DATA_PATH / "outputs"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
OUTPUTS_PATH.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(FORK_PATH))
from cli import build_pipeline, load_yaml

print("DATA_PATH:", DATA_PATH)
print("SPLITS_PATH:", SPLITS_PATH)
print("PROCESSED_PATH:", PROCESSED_PATH)
print("OUTPUTS_PATH:", OUTPUTS_PATH)
print("FORK_PATH:", FORK_PATH)


In [ ]:
# =====================
# Parámetros de ejecución
# =====================
import os
import json

PROFILES = ["core", "py"]
TEXT_COL = "texto"
WINDOW_TOKENS = 12
BATCH_SIZE = 128
N_PROCESS = 1

MODEL_KEY_TO_HF = {
    'beto': 'dccuchile/bert-base-spanish-wwm-cased',
    'roberta_clinical': 'PlanTL-GOB-ES/roberta-base-biomedical-clinical-es',
    'roberta_biomedical': 'PlanTL-GOB-ES/roberta-base-biomedical-es',
}
MODEL_ALIAS = {
    'beto': 'beto',
    'roberta_clinical': 'roberta_clinical',
    'roberta_biomedical': 'roberta_biomedical',
    'BETO': 'beto',
    'ROBERTA_CLINICAL': 'roberta_clinical',
    'ROBERTA_BIOMEDICAL': 'roberta_biomedical',
    'roberta-base-biomedical-clinical-es': 'roberta_clinical',
    'roberta-base-biomedical-es': 'roberta_biomedical',
}

def _normalizar_backbone(raw: str) -> str | None:
    if raw is None:
        return None
    s = str(raw).strip()
    if not s:
        return None
    return MODEL_ALIAS.get(s, MODEL_ALIAS.get(s.lower(), None))

def _resolver_artefacto_seleccion_transformer() -> tuple[dict, Path | None]:
    latest = OUTPUTS_PATH / 'transformer_baseline_selection_latest.json'
    if latest.exists():
        try:
            return json.loads(latest.read_text(encoding='utf-8')), latest
        except Exception:
            pass

    cands = sorted(OUTPUTS_PATH.glob('transformer_baseline_selection_*.json'), key=lambda p: p.stat().st_mtime)
    for p in reversed(cands):
        try:
            return json.loads(p.read_text(encoding='utf-8')), p
        except Exception:
            continue

    return {}, None

# Opciones de ejecución por entorno
RUN_ID = os.getenv('FE_RUN_ID') or pd.Timestamp.now().strftime('fe_%Y%m%d_%H%M%S')
SAMPLE_N = int(os.getenv('FE_SAMPLE_N', '0'))
COMPUTE_SENTIMENT = os.getenv('FE_COMPUTE_SENTIMENT', '1') == '1'
COMPUTE_CONTEXT = os.getenv('FE_COMPUTE_CONTEXT', os.getenv('FE_COMPUTE_BETO', '1')) == '1'
CACHE_KEY = os.getenv('FE_CACHE_KEY', '').strip() or RUN_ID

FE_TEXT_BACKBONE = os.getenv('FE_TEXT_BACKBONE', 'beto').strip().lower()
if FE_TEXT_BACKBONE not in {'auto', 'beto', 'roberta_clinical', 'roberta_biomedical'}:
    raise ValueError('FE_TEXT_BACKBONE inválido. Use: auto, beto, roberta_clinical o roberta_biomedical.')

selection_payload, selection_path = _resolver_artefacto_seleccion_transformer()
selected_from_04c = _normalizar_backbone((selection_payload.get('mejor_transformer_baseline', {}) or {}).get('modelo'))

if FE_TEXT_BACKBONE == 'auto':
    TEXT_BACKBONE = selected_from_04c or 'beto'
    TEXT_BACKBONE_SOURCE = '04c_selection' if selected_from_04c else 'fallback_beto'
else:
    TEXT_BACKBONE = FE_TEXT_BACKBONE
    TEXT_BACKBONE_SOURCE = 'default_beto' if FE_TEXT_BACKBONE == 'beto' else 'env_override'

TEXT_BACKBONE_HF = MODEL_KEY_TO_HF[TEXT_BACKBONE]
CONTEXT_PREFIX = f'ctx_{TEXT_BACKBONE}_'

# Alias legacy para compatibilidad aguas abajo
COMPUTE_BETO = COMPUTE_CONTEXT

# Control explícito de LLM
FE_USE_LLM = os.getenv('FE_USE_LLM', 'auto').strip().lower()
if FE_USE_LLM not in {'auto', '1', '0'}:
    raise ValueError('FE_USE_LLM inválido. Use: auto, 1 o 0.')

print('RUN_ID:', RUN_ID)
print('SAMPLE_N:', SAMPLE_N)
print('COMPUTE_SENTIMENT:', COMPUTE_SENTIMENT)
print('COMPUTE_CONTEXT:', COMPUTE_CONTEXT)
print('TEXT_BACKBONE:', TEXT_BACKBONE, '| source:', TEXT_BACKBONE_SOURCE)
print('CONTEXT_PREFIX:', CONTEXT_PREFIX)
print('CACHE_KEY:', CACHE_KEY)
print('FE_USE_LLM:', FE_USE_LLM)

DENOISED_PATH = DATA_PATH / 'dataset_denoised.csv'
if not DENOISED_PATH.exists():
    raise FileNotFoundError(f'No existe {DENOISED_PATH}. Ejecuta 03 primero.')

GEMINI_JSON = PROCESSED_PATH / 'gemini_extraction.json'
GEMINI_EXISTS = GEMINI_JSON.exists()

if FE_USE_LLM == 'auto':
    USE_LLM = GEMINI_EXISTS
    LLM_MODO_EFECTIVO = 'auto'
elif FE_USE_LLM == '1':
    if not GEMINI_EXISTS:
        raise FileNotFoundError(f'FE_USE_LLM=1 requiere {GEMINI_JSON}, pero no existe.')
    USE_LLM = True
    LLM_MODO_EFECTIVO = 'forzado_on'
else:
    USE_LLM = False
    LLM_MODO_EFECTIVO = 'forzado_off'

APPLY_LLM_TO_PROFILES = {'core', 'py'}
print('USE_LLM:', USE_LLM, '| GEMINI_JSON:', GEMINI_JSON, '| GEMINI_EXISTS:', GEMINI_EXISTS)

CACHE_DIR = PROCESSED_PATH / f'{CACHE_KEY}_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
sent_cache_path = CACHE_DIR / 'sentiment_cache.parquet'
ctx_cache_path = CACHE_DIR / f'{TEXT_BACKBONE}_cls_cache.parquet'

def rules_cache_path(profile: str) -> Path:
    d = PROCESSED_PATH / f'{RUN_ID}_{profile}'
    d.mkdir(parents=True, exist_ok=True)
    return d / f'rule_matrix_{profile}.parquet'

def features_out_dir(profile: str) -> Path:
    d = PROCESSED_PATH / f'{RUN_ID}_{profile}'
    d.mkdir(parents=True, exist_ok=True)
    return d


## 1) Cargar dataset denoised y construir `df_base_feat`

`df_base_feat` es la tabla base con:
- identificadores (`row_id`, `patient_id`, `etiqueta`)
- flags de denoising
- **sentimiento**
- **embeddings contextuales `ctx_<backbone>_*`**

Esta parte es independiente del perfil (`core` / `py`) y se puede cachear.


In [ ]:
df = pd.read_csv(DENOISED_PATH)
if SAMPLE_N > 0:
    df = df.head(SAMPLE_N).copy()

if TEXT_COL not in df.columns:
    raise ValueError(f"Columna {TEXT_COL} no existe en {DENOISED_PATH}. Columnas: {list(df.columns)[:30]}")

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

if "row_id" not in df.columns:
    df = df.reset_index(drop=True)
    df.insert(0, "row_id", np.arange(len(df), dtype=int))

text_col = guess_text_col(df)
print("dataset_denoised rows:", len(df), "| cols:", len(df.columns))
print("text_col:", text_col)


In [ ]:
# =====================
# Sentimiento y embeddings contextuales (cachés)
# =====================
CONTEXT_BATCH_SIZE = 16
CONTEXT_MAX_LENGTH = 256

print('sent_cache exists:', sent_cache_path.exists())
print('ctx_cache exists:', ctx_cache_path.exists())

def _merge_cache(df_input: pd.DataFrame, df_cache: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in df_cache.columns if c != 'row_id']
    base = df_input.drop(columns=[c for c in cols if c in df_input.columns], errors='ignore')
    return base.merge(df_cache, on='row_id', how='left')

def attach_sentiment(df_input: pd.DataFrame, text_col: str = 'texto') -> pd.DataFrame:
    if sent_cache_path.exists():
        cache = pd.read_parquet(sent_cache_path)
        print(f'[CACHE] Sentimiento: {sent_cache_path.name}')
        return _merge_cache(df_input, cache)

    try:
        from pysentimiento import create_analyzer
    except ImportError as e:
        raise ImportError('Falta pysentimiento. Instala con: pip install pysentimiento') from e

    analyzer = create_analyzer(task='sentiment', lang='es')
    rows = []
    for rid, text in tqdm(
        df_input[['row_id', text_col]].itertuples(index=False),
        total=len(df_input),
        desc='Sentimiento',
    ):
        pred = analyzer.predict(str(text))
        probas = getattr(pred, 'probas', {}) or {}
        rows.append({
            'row_id': int(rid),
            'sent_pos': float(probas.get('POS', 0.0)),
            'sent_neu': float(probas.get('NEU', 0.0)),
            'sent_neg': float(probas.get('NEG', 0.0)),
        })

    cache = pd.DataFrame(rows)
    cache.to_parquet(sent_cache_path, index=False)
    print(f'Cache de sentimiento guardada: {sent_cache_path}')
    return _merge_cache(df_input, cache)

def attach_context_cls(df_input: pd.DataFrame, text_col: str = 'texto') -> pd.DataFrame:
    if ctx_cache_path.exists():
        cache = pd.read_parquet(ctx_cache_path)
        print(f'[CACHE] Context CLS ({TEXT_BACKBONE}): {ctx_cache_path.name}')
        return _merge_cache(df_input, cache)

    try:
        import torch
        from transformers import AutoTokenizer, AutoModel
    except ImportError as e:
        raise ImportError('Faltan torch/transformers. Instala con: pip install torch transformers') from e

    device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
    tokenizer = AutoTokenizer.from_pretrained(TEXT_BACKBONE_HF)
    model = AutoModel.from_pretrained(TEXT_BACKBONE_HF).to(device)
    model.eval()

    rows = []
    texts = df_input[text_col].astype(str).tolist()
    row_ids = df_input['row_id'].astype(int).tolist()

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), CONTEXT_BATCH_SIZE), desc=f'CTX CLS ({TEXT_BACKBONE})'):
            batch_texts = texts[i:i + CONTEXT_BATCH_SIZE]
            batch_ids = row_ids[i:i + CONTEXT_BATCH_SIZE]

            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=CONTEXT_MAX_LENGTH,
                return_tensors='pt',
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            out = model(**enc)
            cls = out.last_hidden_state[:, 0, :].detach().cpu().numpy().astype('float32')

            for rid, vec in zip(batch_ids, cls):
                r = {'row_id': int(rid)}
                for j, val in enumerate(vec):
                    r[f'{CONTEXT_PREFIX}{j:03d}'] = float(val)
                rows.append(r)

    cache = pd.DataFrame(rows)
    cache.to_parquet(ctx_cache_path, index=False)
    print(f'Cache contextual guardada: {ctx_cache_path}')
    return _merge_cache(df_input, cache)


In [ ]:
# Aplicar cachés al conjunto de datos base una sola vez
df_base_feat = df.copy()

if COMPUTE_SENTIMENT:
    df_base_feat = attach_sentiment(df_base_feat, text_col=text_col)

if COMPUTE_CONTEXT:
    df_base_feat = attach_context_cls(df_base_feat, text_col=text_col)

print('df_base_feat cols:', len(df_base_feat.columns))
print('has sent:', any(c.startswith('sent_') for c in df_base_feat.columns))
print('has ctx:', any(c.startswith(CONTEXT_PREFIX) for c in df_base_feat.columns))


## 2) Extracción de reglas (`rule_*`, `niega_*`) por perfil

Esta sección:
- calcula la matriz de reglas por perfil (`core`, `py`)
- guarda un cache tabular `rule_matrix_<profile>.parquet`

👉 Si después aparece el JSON del LLM, **no hace falta recalcular reglas**.  
Puedes volver directo a la sección de integración LLM + export.


In [ ]:
FENOS_CFG = load_yaml(FORK_PATH / "configs" / "fenotipos.yml")

def extract_rule_matrix(nlp, df, text_col, batch_size=128, n_process=1, window_tokens=12, add_patient_neg_features=True):
    texts = df[text_col].tolist()
    row_ids = df["row_id"].tolist()

    rule_rows = []
    seen_cats = set()

    for rid, doc in tqdm(zip(row_ids, nlp.pipe(texts, batch_size=batch_size, n_process=n_process)),
                         total=len(row_ids), desc="Reglas"):
        d = {"row_id": int(rid)}
        for ent in getattr(doc, "ents", []):
            keep, is_pat_neg = keep_entity(ent, doc, window_tokens=window_tokens)
            if not keep:
                continue
            cat = ent.label_
            d[f"rule_{cat}"] = 1
            seen_cats.add(cat)
            if add_patient_neg_features and is_pat_neg:
                d[f"niega_{cat}"] = 1
        rule_rows.append(d)

    df_rules = pd.DataFrame(rule_rows).fillna(0)

    # Asegurar columnas para todas las categorías vistas
    for cat in sorted(seen_cats):
        rc = f"rule_{cat}"
        if rc not in df_rules.columns:
            df_rules[rc] = 0
        if add_patient_neg_features:
            nc = f"niega_{cat}"
            if nc not in df_rules.columns:
                df_rules[nc] = 0

    for c in df_rules.columns:
        if c != "row_id":
            df_rules[c] = df_rules[c].astype(int)

    rule_cols = [c for c in df_rules.columns if c.startswith("rule_")]
    niega_cols = [c for c in df_rules.columns if c.startswith("niega_")]
    df_rules["n_rules"] = df_rules[rule_cols].sum(axis=1)
    df_rules["feat_negacion_paciente"] = (df_rules[niega_cols].sum(axis=1) > 0).astype(int)

    print(f"Matriz reglas: n_cols={len(df_rules.columns)} | rule_cols={len(rule_cols)} | niega_cols={len(niega_cols)} | any_rule={(df_rules['n_rules']>0).mean():.1%} | avg_rules={df_rules['n_rules'].mean():.2f}")
    return df_rules

# Ejecutar o cargar caché de reglas
for profile in PROFILES:
    rpath = rules_cache_path(profile)
    if rpath.exists():
        print(f"[CACHE] Cargando reglas {profile}: {rpath.name}")
    else:
        print("\n" + "="*80)
        print("EXTRACT PROFILE:", profile)
        nlp = build_pipeline(profile, FENOS_CFG)
        df_rules = extract_rule_matrix(
            nlp=nlp,
            df=df,
            text_col=TEXT_COL,
            batch_size=BATCH_SIZE,
            n_process=N_PROCESS,
            window_tokens=WINDOW_TOKENS,
            add_patient_neg_features=True,
        )
        df_rules = df_rules.loc[:, ~df_rules.columns.duplicated()].copy()
        df_rules.to_parquet(rpath, index=False)
        print("Cache de reglas guardada:", rpath)


## 3) Integración LLM (celda separada, opcional)

Esta es la sección importante para tu flujo:

- Si **no existe** `data/processed/gemini_extraction.json`, no pasa nada.
- Si ya existe, puedes ejecutar **solo esta sección y la de export**.
- **No recalcula reglas**, porque usa el cache tabular por perfil.
- **No recalcula contexto/sentimiento**, porque usa `df_base_feat`.

La fusión es **Late Fusion lógica**:
- `feat_<Fenotipo> = max(rule_<Fenotipo>, llm_<Fenotipo>)`
- `feat_niega_<Fenotipo> = max(niega_<Fenotipo>, llm_niega_<Fenotipo>)`

Las columnas `llm_*` son temporales y se eliminan por defecto.


In [ ]:
def llm_late_fusion_inplace(
    df_rules: pd.DataFrame,
    gemini_json_path,
    rule_prefix: str = "rule_",
    niega_prefix: str = "niega_",
    feat_prefix: str = "feat_",
    drop_original_rule_cols: bool = False,
    drop_llm_cols: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    gemini_json_path = Path(gemini_json_path)

    if "row_id" not in df_rules.columns:
        raise ValueError("df_rules debe contener la columna 'row_id'.")
    if not gemini_json_path.exists():
        raise FileNotFoundError(f"No existe el archivo LLM: {gemini_json_path}")

    out = df_rules.copy()

    # Categorías para `late fusion` tomadas de `rule_*`; `medication_*` se preserva separado y no se fusiona diagnósticamente con LLM
    cats = sorted([
        c.replace(rule_prefix, "", 1)
        for c in out.columns
        if c.startswith(rule_prefix)
        and not c.startswith(f"{rule_prefix}medication_")
    ])

    if verbose:
        print(f"[LLM] Categorías candidatas a fusión: {len(cats)}")
        print(f"[LLM] Ejemplos: {cats[:10]}")

    for cat in cats:
        ncol = f"{niega_prefix}{cat}"
        if ncol not in out.columns:
            out[ncol] = 0

    with open(gemini_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        rows = []
        for k, v in data.items():
            if isinstance(v, dict):
                row = {"row_id": int(k)}
                row.update(v)
            else:
                row = {"row_id": int(k), "sintomas": v}
            rows.append(row)
        df_g = pd.DataFrame(rows)
    else:
        df_g = pd.DataFrame(data)

    if df_g.empty:
        if verbose:
            print("[LLM] gemini_extraction.json vacío → no se fusiona")
        for cat in cats:
            out[f"{feat_prefix}{cat}"] = out[f"{rule_prefix}{cat}"].astype(int)
            out[f"{feat_prefix}niega_{cat}"] = out[f"{niega_prefix}{cat}"].astype(int)
        return out

    if "row_id" not in df_g.columns:
        raise ValueError("gemini_extraction.json debe contener 'row_id'.")
    if "sintomas" not in df_g.columns and "sintomas_mapeados" in df_g.columns:
        df_g["sintomas"] = df_g["sintomas_mapeados"]
    if "sintomas" not in df_g.columns:
        raise ValueError("gemini_extraction.json no contiene 'sintomas' ni 'sintomas_mapeados'.")

    df_g = df_g[["row_id", "sintomas"]].copy()
    df_g["sintomas"] = df_g["sintomas"].apply(lambda x: x if isinstance(x, list) else [])
    df_g = df_g.explode("sintomas").dropna(subset=["sintomas"]).rename(columns={"sintomas": "sintoma"})
    df_g["sintoma"] = df_g["sintoma"].astype(str).str.strip()
    df_g = df_g[df_g["sintoma"] != ""]

    # Normalización robusta sin regex frágil
    df_g["sintoma_l"] = df_g["sintoma"].str.lower()
    df_g["is_neg"] = df_g["sintoma_l"].str.startswith("niega_")
    df_g["base_sym"] = df_g["sintoma_l"].str.replace("niega_", "", regex=False)

    cat_map = {c.lower(): c for c in cats}
    df_g["cat"] = df_g["base_sym"].map(cat_map)

    before = len(df_g)
    df_g = df_g[df_g["cat"].notna()].copy()
    after = len(df_g)
    if verbose:
        print(f"[LLM] Hallazgos antes de filtrar ontología: {before}")
        print(f"[LLM] Hallazgos mapeados a ontología:     {after}")

    if len(df_g) > 0:
        df_pos = (
            df_g[~df_g["is_neg"]]
            .assign(val=1)
            .pivot_table(index="row_id", columns="cat", values="val", aggfunc="max", fill_value=0)
            .add_prefix("llm_")
            .reset_index()
        )
        df_neg = (
            df_g[df_g["is_neg"]]
            .assign(val=1)
            .pivot_table(index="row_id", columns="cat", values="val", aggfunc="max", fill_value=0)
            .add_prefix("llm_niega_")
            .reset_index()
        )

        if "row_id" in df_pos.columns and "row_id" in df_neg.columns:
            df_llm = df_pos.merge(df_neg, on="row_id", how="outer").fillna(0)
        elif "row_id" in df_pos.columns:
            df_llm = df_pos.copy()
        elif "row_id" in df_neg.columns:
            df_llm = df_neg.copy()
        else:
            df_llm = pd.DataFrame({"row_id": out["row_id"].unique()})
    else:
        df_llm = pd.DataFrame({"row_id": out["row_id"].unique()})

    out = out.merge(df_llm, on="row_id", how="left").fillna(0)

    for cat in cats:
        rcol = f"{rule_prefix}{cat}"
        ncol = f"{niega_prefix}{cat}"
        lcol = f"llm_{cat}"
        lncol = f"llm_niega_{cat}"

        if rcol not in out.columns: out[rcol] = 0
        if ncol not in out.columns: out[ncol] = 0
        if lcol not in out.columns: out[lcol] = 0
        if lncol not in out.columns: out[lncol] = 0

        out[f"{feat_prefix}{cat}"] = out[[rcol, lcol]].max(axis=1).astype(int)
        out[f"{feat_prefix}niega_{cat}"] = out[[ncol, lncol]].max(axis=1).astype(int)

    if drop_llm_cols:
        llm_cols = [c for c in out.columns if c.startswith("llm_")]
        out = out.drop(columns=llm_cols, errors="ignore")

    if drop_original_rule_cols:
        orig_cols = [f"{rule_prefix}{cat}" for cat in cats] + [f"{niega_prefix}{cat}" for cat in cats]
        out = out.drop(columns=[c for c in orig_cols if c in out.columns], errors="ignore")

    feat_niega_cols = [f"{feat_prefix}niega_{cat}" for cat in cats if f"{feat_prefix}niega_{cat}" in out.columns]
    if feat_niega_cols:
        out["feat_negacion_paciente"] = (out[feat_niega_cols].sum(axis=1) > 0).astype(int)

    if verbose:
        n_feat = len([c for c in out.columns if c.startswith(feat_prefix) and not c.startswith(f"{feat_prefix}niega_")])
        n_feat_neg = len([c for c in out.columns if c.startswith(f"{feat_prefix}niega_")])
        print(f"[LLM] Late fusion completada | feat_*={n_feat} | feat_niega_*={n_feat_neg} | shape={out.shape}")

    return out


In [ ]:
# =====================
# Integración LLM y exportación por perfil
# =====================
# Puede re-ejecutarse solo esta celda cuando aparezca `gemini_extraction.json`.
# La estrategia es `late fusion`: `feat_* = max(rule_*, llm_*)` únicamente para síntomas.

# No recalcula reglas ni embeddings contextuales ni sentimiento.

summary_rows = []

for profile in PROFILES:
    print('\n' + '='*80)
    print('EXPORT PROFILE:', profile)

    rpath = rules_cache_path(profile)
    if not rpath.exists():
        raise FileNotFoundError(f'Falta el rule cache de {profile}: {rpath}. Ejecuta primero la sección de reglas.')

    df_rules = pd.read_parquet(rpath)
    df_rules = df_rules.loc[:, ~df_rules.columns.duplicated()].copy()

    if USE_LLM and (profile in APPLY_LLM_TO_PROFILES):
        print(f'[LLM] Aplicando late fusion a profile={profile}')
        df_rules = llm_late_fusion_inplace(
            df_rules=df_rules,
            gemini_json_path=GEMINI_JSON,
            rule_prefix='rule_',
            niega_prefix='niega_',
            feat_prefix='feat_',
            drop_original_rule_cols=False,
            drop_llm_cols=True,
            verbose=True,
        )
    else:
        print(f'[LLM] No se aplica LLM para profile={profile}.')
        rule_cols = [c for c in df_rules.columns if c.startswith('rule_') and not c.startswith('rule_medication_')]
        for rc in rule_cols:
            ph = rc.replace('rule_', '', 1)
            if f'feat_{ph}' not in df_rules.columns:
                df_rules[f'feat_{ph}'] = df_rules[rc].astype(int)
            nc = f'niega_{ph}'
            if nc not in df_rules.columns:
                df_rules[nc] = 0
            if f'feat_niega_{ph}' not in df_rules.columns:
                df_rules[f'feat_niega_{ph}'] = df_rules[nc].astype(int)

        feat_niega_cols = [c for c in df_rules.columns if c.startswith('feat_niega_')]
        if 'feat_negacion_paciente' not in df_rules.columns:
            df_rules['feat_negacion_paciente'] = (df_rules[feat_niega_cols].sum(axis=1) > 0).astype(int)

    base = df_base_feat.copy()
    if 'feat_negacion_paciente' in base.columns:
        base = base.drop(columns=['feat_negacion_paciente'])

    df_feat = base.merge(df_rules.drop(columns=['n_rules'], errors='ignore'), on='row_id', how='left')

    bin_prefixes = ('rule_', 'niega_', 'feat_', 'llm_')
    for c in [c for c in df_feat.columns if c.startswith(bin_prefixes)]:
        df_feat[c] = df_feat[c].fillna(0).astype(int)

    out_dir = features_out_dir(profile)
    p_parquet = out_dir / f'features_{profile}.parquet'
    p_csv = out_dir / f'features_{profile}.csv'

    df_feat.to_parquet(p_parquet, index=False)
    df_feat.to_csv(p_csv, index=False)

    n_rule = sum(c.startswith('rule_') for c in df_feat.columns)
    n_niega = sum(c.startswith('niega_') for c in df_feat.columns)
    n_feat = sum(c.startswith('feat_') and not c.startswith('feat_niega_') for c in df_feat.columns)
    n_feat_niega = sum(c.startswith('feat_niega_') for c in df_feat.columns)
    n_sent = sum(c.startswith('sent_') for c in df_feat.columns)
    n_ctx = sum(c.startswith(CONTEXT_PREFIX) for c in df_feat.columns)

    print('Parquet exportado:', p_parquet)
    print('CSV exportado:', p_csv)
    print(f'n_rule={n_rule} | n_niega={n_niega} | n_feat={n_feat} | n_feat_niega={n_feat_niega} | n_sent={n_sent} | n_ctx={n_ctx}')

    summary_rows.append({
        'profile': profile,
        'run_id': RUN_ID,
        'rows': len(df_feat),
        'llm_activo': int(USE_LLM),
        'sentimiento_activo': int(COMPUTE_SENTIMENT),
        'contexto_activo': int(COMPUTE_CONTEXT),
        'beto_activo': int(COMPUTE_CONTEXT),  # alias legacy
        'text_backbone': TEXT_BACKBONE,
        'context_prefix': CONTEXT_PREFIX,
        'input_path': str(DENOISED_PATH),
        'gemini_path': str(GEMINI_JSON),
        'gemini_exists': int(GEMINI_EXISTS),
        'n_rule': n_rule,
        'n_niega': n_niega,
        'n_feat': n_feat,
        'n_feat_niega': n_feat_niega,
        'n_sent': n_sent,
        'n_ctx': n_ctx,
        'path': str(p_parquet),
    })

df_summary = pd.DataFrame(summary_rows)
summary_path = PROCESSED_PATH / f'{RUN_ID}_feature_summary.csv'
df_summary.to_csv(summary_path, index=False)
print('\nResumen de features:', summary_path)

cfg = {
    'run_id': RUN_ID,
    'timestamp': pd.Timestamp.now().isoformat(),
    'input_path': str(DENOISED_PATH),
    'sample_n': SAMPLE_N,
    'profiles': PROFILES,
    'llm_modo_solicitado': FE_USE_LLM,
    'llm_modo_efectivo': LLM_MODO_EFECTIVO,
    'llm_activo': bool(USE_LLM),
    'gemini_path': str(GEMINI_JSON),
    'gemini_exists': bool(GEMINI_EXISTS),
    'sentimiento_activo': bool(COMPUTE_SENTIMENT),
    'contexto_activo': bool(COMPUTE_CONTEXT),
    'beto_activo': bool(COMPUTE_CONTEXT),
    'text_backbone': TEXT_BACKBONE,
    'text_backbone_source': TEXT_BACKBONE_SOURCE,
    'text_backbone_model_hf': TEXT_BACKBONE_HF,
    'context_prefix': CONTEXT_PREFIX,
    'transformer_selection_path': str(selection_path) if selection_path else None,
    'cache_key': CACHE_KEY,
    'cache_dir': str(CACHE_DIR),
    'feature_summary_csv': str(summary_path),
}
cfg_path = PROCESSED_PATH / f'{RUN_ID}_config.json'
with open(cfg_path, 'w', encoding='utf-8') as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)
print('Configuración de corrida:', cfg_path)

df_summary
